# NFL Game Simulator v6

**Files in one folder:** `nfl_game_sim_v6.py`, `nfl_props_v6.py`, `nfl_weekly_v6.py`, `fetch_dk_props.py`, `run_week.py`, this notebook, and (optional) `odds_api_key.txt` with your Odds API key. After replacing any `.py` file: **Kernel → Restart**.

**What's new in v6**
- **One command per week.** `weekly_report()` refreshes this season's data, simulates every game and every player, pulls DraftKings' lines, logs the predictions, and saves `reports/<season>_wk<week>.xlsx` and `.html`. From a terminal: `python run_week.py --week 5`.
- **Prediction log + closing line value.** Every run logs predictions with the market line at that moment (`logs/`). `grade_log()` grades them after the games, including whether the market moved toward the model's number before kickoff.
- **Consistent props.** In every simulated game, the QB's completions, yards, and TDs are the sum of what his receivers did, and yards rise and fall with that game's score. This makes same-game combos priceable (section 4).
- **Scoring level.** Tested, and not applied by default: it made over/under picks worse in backtests. The dashboard reports the league's scoring level for context.

Each cell below depends only on `report` (section 1) or `g` (section 2), so they can be re-run in any order after those two.

In [1]:
# First time only
# %pip install pandas numpy pyarrow openpyxl matplotlib

In [2]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from nfl_weekly_v6 import *          # also gives you ns (game model) and props (props model)
pd.set_option("display.max_rows", 200)

## 1. Run the week
Re-run this cell whenever you want fresh data or lines: after Wednesday's first injury report, after Friday's final report, and Sunday morning after inactives. Every run is logged, so the log keeps the latest pre-kickoff prediction.

In [3]:
SEASON, WEEK = 2026, 4
REFRESH = True        # re-download this season's data and pull ESPN's live injury list
REFRESH_HOURS = 12    # files older than this re-download; 0 = force everything fresh (IR news, roster moves)
WEATHER = {}          # e.g. {"BUF": {"temp": 31, "wind": 22}}  (keyed by home team)
REFEREES = {}         # e.g. {"PHI": "Shawn Hochuli"}          (keyed by home team)
N_SIMS = 10000        # simulations per game for the full week (more = smoother, slower)
LINES = "auto"        # "auto" = Odds API if you have a key, else FanDuel, else ESPN | "fanduel" | "espn" | "odds_api"

report = weekly_report(week=WEEK, season=SEASON, refresh=REFRESH, refresh_hours=REFRESH_HOURS,
                       weather=WEATHER, referees=REFEREES, n_sims=N_SIMS, lines=LINES)

Data is fresh (no download needed).
No odds_api lines; trying fanduel.
  ATL @ NO: 99 FanDuel prices
Wrote 99 FanDuel prices for 1 games to fd_props_current.csv

2026 week 4 | Week 4 injury report; live ESPN injury list as of Mon Oct 05, 09:48 AM Arizona time has the final word
  matchup  home_win_pct  model_line  vegas_line  model_total  vegas_total
PIT @ CLE          54.6        -1.7         2.5         42.0         38.5
IND @ WAS          36.9         4.3         4.5         49.3         46.5
TEN @ BAL          65.1        -5.3       -11.5         45.0         42.5
 NE @ BUF          60.8        -3.9        -7.0         51.6         49.5
NYJ @ CHI          72.0        -8.0        -3.5         44.1         43.5
JAX @ CIN          39.0         3.6        -2.5         49.9         51.5
DAL @ HOU          53.6        -1.4        -3.0         52.4         48.5
ARI @ NYG          65.6        -5.4         2.5         46.1         44.5
 LA @ PHI          49.8        -0.2         3.5        

In [4]:
games_table(report)

,Matchup,QBs (away / home),Home win %,Model line,Vegas line,Model's side,Spread gap,Model total,Vegas total,Model's total lean,Total gap,Final
0,PIT @ CLE,A.Rodgers / D.Watson,54.6,-1.7,2.5,CLE +2.5,4.2,42.0,38.5,Over 38.5,3.5,PIT 24 - CLE 27
1,IND @ WAS,D.Jones / Athan Kaliakmanis (85%) / M.Mariota ...,36.9,4.3,4.5,WAS +4.5,0.2,49.3,46.5,Over 46.5,2.8,IND 30 - WAS 13
2,TEN @ BAL,C.Ward / T.Huntley (55%) / L.Jackson (45%),65.1,-5.3,-11.5,TEN +11.5,6.2,45.0,42.5,Over 42.5,2.5,TEN 18 - BAL 24
3,NE @ BUF,D.Maye / J.Allen,60.8,-3.9,-7.0,NE +7,3.1,51.6,49.5,Over 49.5,2.1,NE 29 - BUF 26
4,NYJ @ CHI,G.Smith / T.Bagent,72.0,-8.0,-3.5,CHI -3.5,4.5,44.1,43.5,Over 43.5,0.6,NYJ 12 - CHI 23
5,JAX @ CIN,T.Lawrence / J.Burrow,39.0,3.6,-2.5,JAX +2.5,6.1,49.9,51.5,Under 51.5,1.6,JAX 22 - CIN 17
6,DAL @ HOU,D.Prescott / C.Stroud,53.6,-1.4,-3.0,DAL +3,1.6,52.4,48.5,Over 48.5,3.9,DAL 34 - HOU 30
7,ARI @ NYG,J.Brissett / J.Winston,65.6,-5.4,2.5,NYG +2.5,7.9,46.1,44.5,Over 44.5,1.6,ARI 24 - NYG 36
8,LA @ PHI,M.Stafford / J.Hurts,49.8,-0.2,3.5,PHI +3.5,3.7,46.0,42.5,Over 42.5,3.5,LA 24 - PHI 20
9,GB @ TB,J.Love / J.Daniels,42.2,2.6,3.0,TB +3,0.4,47.4,38.5,Over 38.5,8.9,GB 17 - TB 14


### Injuries the model is using
Official reports come out Wed-Fri, so the model also reads ESPN's live injury list (updates within hours of news) and treats it as the final word. A starting QB who's uncertain (Questionable/Doubtful) is simulated both ways, in proportion to his chance of playing. Check this table when a line looks off; override with `out=[...]` or `home_qb=` in section 2.

In [5]:
report['injuries']

,Team,Player,Pos,Status used,Chance out,Latest news
0,ARI,Starling Thomas V,CB,Out (ESPN Oct 04: Not Specified Coach's Decision),100%,inactive
1,ARI,Dadrion Taylor-Demerson,S,Out (ESPN Oct 04: Not Specified Back),100%,inactive
2,ARI,Josh Fryar,OT,Out (ESPN Oct 04: Not Specified Coach's Decision),100%,inactive
3,ARI,Reggie Virgil,WR,Out (ESPN Oct 04: Not Specified Coach's Decision),100%,Virgil (coach's decision) is inactive for Sund...
4,ARI,Carson Beck,QB,Out (ESPN Oct 04: Not Specified Coach's Decision),100%,Beck (coach's decision) is inactive and is des...
...,...,...,...,...,...,...
305,WAS,Jaylin Lane,WR,Out (ESPN Oct 05: Surgery Ankle),100%,Lane was diagnosed with a fracture and disloca...
306,WAS,Marcus Mariota,QB,Doubtful (ESPN Oct 04: Sprain Knee - MCL),85%,"Mariota (knee), who was ruled out for the rema..."
307,WAS,Tyler Owens,S,Questionable (ESPN Oct 04: Not Specified Hamst...,25%,Owens (hamstring) is questionable to return to...
308,WAS,Jeremy Reaves,S,Questionable (ESPN Oct 04: Not Specified Finger),25%,Reaves (finger) is questionable to return to S...


### Recent IR / Out moves
Everyone ESPN has put on IR, PUP, Out, or suspension since `SINCE` (None = last 4 days). `In model = True` means the simulations have him ruled out. If a move happened after your last run, re-run section 1 with `REFRESH_HOURS = 0`.

In [6]:
SINCE = None          # e.g. "2026-09-28"
ir_moves(report, since=SINCE)

ESPN injury list as of Mon Oct 05 09:48 AM Arizona time; 207 players listed since Oct 01.


,Player,Team,Pos,Status,Listed (AZ),Injury,In model
0,Chris Moore,BAL,WR,Out,Mon Oct 05 09:41 AM,Not Specified Ankle,True
1,Caleb Williams,CHI,QB,Out,Mon Oct 05 09:30 AM,Strain Hamstring,True
2,Anthony Johnson Jr.,CHI,S,Out,Mon Oct 05 09:27 AM,Not Specified Undisclosed,True
3,Jaylin Lane,WAS,WR,Out,Mon Oct 05 08:27 AM,Surgery Ankle,True
4,Jordyn Tyson,NO,WR,Injured Reserve,Mon Oct 05 06:36 AM,Strain Hamstring,True
...,...,...,...,...,...,...,...
202,Rico Dowdle,PIT,RB,Out,Thu Oct 01 03:51 PM,Not Specified Toe,True
203,Kevin Jobity Jr.,PIT,DT,Out,Thu Oct 01 03:51 PM,Not Specified Coach's Decision,True
204,Gabriel Rubio,PIT,DE,Out,Thu Oct 01 03:51 PM,Not Specified Coach's Decision,False
205,Justin Jefferson,CLE,LB,Out,Thu Oct 01 03:48 PM,Not Specified Coach's Decision,True


## 2. Explore one game with your own settings
`GAME_SIMS` sets how many times the game is played (every stat and prop is averaged across all of them). `home_qb` / `away_qb` force a starter, `out` rules players out ("C.Williams" or "Caleb Williams").

In [7]:
GAME_SIMS = 20000     # how many times to play this game (10,000-50,000; more = smoother %s, slower)

g = explore_game(report, home="CHI", away="NYJ",
                 home_qb=None, away_qb=None, out=[],
                 weather=None, referee=None, n_sims=GAME_SIMS)
print(f"Simulated {len(g.game['home_scores']):,} games.")

NYJ (G.Smith) @ CHI (T.Bagent)
  Win prob:    CHI 72.5% | NYJ 26.7% | tie 0.7%
  Final proj:  CHI 25.9 - NYJ 17.9   (halftime 12.6 - 7.1)
  Line:        CHI -8.0   Total 43.9
  Possessions: CHI 11.1, NYJ 11.0   Turnovers: CHI 1.27, NYJ 1.39
  Most likely finals: CHI 27 - NYJ 20 (0.8%), CHI 24 - NYJ 17 (0.8%), CHI 20 - NYJ 13 (0.7%)
  Conditions:  72F, wind 3 mph (total -0.1) | rest: CHI 6 days, NYJ 7 days | NYJ crosses 1 time zone(s) (CHI -0.1)
  Referee:     Carl Cheffers: 12.5 flags/game (league 12.0), away-minus-home +0.48 (120 games) (CHI +0.00)
  CHI offense: C.Santos (54/64, +0.00 vs avg) | QB pressured 13.1% of dropbacks (league 15.4%) | red-zone TD 58.7% (league 58.1%)
  NYJ offense: J.Sanders (42/47, +0.13 vs avg) | QB pressured 14.4% of dropbacks (league 15.4%) | red-zone TD 58.4% (league 58.1%)
  NYJ defensive starters out: Kingsley Enagbare (DL, Out (ESPN Oct 04: Not Specified Knee))
  CHI defensive starters out: Cameron Lewis (CB, Out (ESPN Oct 04: Not Specified Hip))
  CH

In [8]:
g.qbs()

,player,team,pass_att,completions,pass_yds,pass_tds,rush_att,rush_yds,anytime_td,td_odds
0,Tyson Bagent,CHI,31.0,19.0,200.7,2.0,0.0,0.0,2%,+5502
1,Geno Smith,NYJ,32.0,21.0,222.3,1.0,5.0,20.1,11%,+833


In [9]:
g.rbs()

,player,team,rush_att,rush_yds,targets,receptions,rec_yds,anytime_td,td_odds
0,D'Andre Swift,CHI,18.0,75.7,2.0,1.0,7.5,53%,-114
1,Kyle Monangai,CHI,8.0,33.2,1.0,1.0,2.4,34%,+193
2,Zavier Scott,CHI,2.0,8.8,1.0,0.0,0.0,14%,+606
3,Braelon Allen,NYJ,8.0,33.5,1.0,1.0,4.0,30%,+235
4,Kene Nwangwu,NYJ,5.0,20.1,0.0,0.0,0.0,24%,+322
5,Isaiah Davis,NYJ,4.0,14.2,1.0,0.0,0.0,11%,+847


In [10]:
g.receivers()

,player,pos,team,targets,receptions,rec_yds,anytime_td,td_odds
0,Luther Burden III,WR,CHI,6.0,3.0,36.8,28%,+258
1,Kalif Raymond,WR,CHI,4.0,3.0,29.7,23%,+336
2,Colston Loveland,TE,CHI,5.0,3.0,28.4,21%,+365
3,Rome Odunze,WR,CHI,4.0,2.0,25.5,16%,+508
4,Garrett Wilson,WR,NYJ,9.0,6.0,67.7,30%,+233
5,Kenyon Sadiq,TE,NYJ,5.0,4.0,33.3,20%,+410
6,Isaiah Williams,WR,NYJ,4.0,3.0,24.7,11%,+845
7,Jeremy Ruckert,TE,NYJ,3.0,2.0,14.4,11%,+834
8,Sterling Shepard,WR,NYJ,2.0,2.0,12.9,8%,+1167
9,Tyler Johnson,WR,NYJ,2.0,1.0,10.3,8%,+1168


## 3. One prop vs a line
`LINE = None` uses DraftKings' current line (needs the API key); type a number to test your own. Probabilities are counted straight from the simulations.

In [11]:
PLAYER, MARKET, LINE = "Garrett Wilson", "rec_yds", None
print(props.prop_check(g.props.medians, report["dk_ou"], report["dk_yes"], PLAYER, MARKET, line=LINE, sims=g.props))
g.props.distribution(PLAYER, MARKET)

Garrett Wilson: projection 67.7 (no DraftKings line found; pass line= to test one)


10th     22.670256
25th     40.855839
50th     67.717492
75th    102.582945
90th    143.221763
dtype: float64

## 4. Same-game combos
Legs: `(player, market, "over"/"under", line)`, `(player, "anytime_td")`, `(team, "win")`, `(team, "spread", -2.5)`, `("GAME", "total", "over", 44.5)`.

`correlation_lift` above 1 = the legs tend to hit together, so the combo is more likely than multiplying the single-leg odds. Put `dk_price="+450"` after the closing `]` (not inside the list) to compare with DraftKings' same-game-parlay price (the Odds API doesn't provide SGP prices, so type it in).

In [12]:
g.combo([("Geno Smith", "pass_yds", "over", 220.5),
         ("Garrett Wilson", "rec_yds", "over", 70.5)])

{'legs': ['Geno Smith pass_yds over 220.5',
  'Garrett Wilson rec_yds over 70.5'],
 'leg_probs': [0.508, 0.476],
 'joint_prob': 0.3534,
 'if_independent': 0.2421,
 'correlation_lift': 1.46,
 'fair_odds': '+183'}

In [13]:
g.combo([("NYJ", "win"),
         ("Garrett Wilson", "anytime_td")], dk_price="+600")

{'legs': ['NYJ win', 'Garrett Wilson anytime_td'],
 'leg_probs': [0.267, 0.301],
 'joint_prob': 0.1075,
 'if_independent': 0.0804,
 'correlation_lift': 1.34,
 'fair_odds': '+831',
 'dk_price': '+600',
 'ev_per_unit': -0.248}

## 5. Most likely TD scorer per game
Shortest fair anytime-TD odds in each game, straight from the simulations. `TOP = 3` lists three per game. DraftKings price and EV per $1 show up when lines were pulled.

In [14]:
TOP = 1
favs = td_favorites(report, top=TOP)
favs.to_csv(f"reports/{SEASON}_wk{WEEK:02d}_td_favorites.csv", index=False)
favs

,Game,Kickoff (AZ),Player,Pos,Team,Model TD %,Fair odds,Expected TDs,Book price,EV per $1
0,PIT @ CLE,Thu 05:15 PM,Jaylen Warren,RB,PIT,60.1,-151,0.83,NaN,NaN
1,IND @ WAS,Sun 06:30 AM,Jonathan Taylor,RB,IND,64.9,-185,0.96,NaN,NaN
2,TEN @ BAL,Sun 10:00 AM,Derrick Henry,RB,BAL,76.5,-325,1.26,NaN,NaN
3,NE @ BUF,Sun 10:00 AM,James Cook,RB,BUF,50.1,-100,0.66,NaN,NaN
4,NYJ @ CHI,Sun 10:00 AM,D'Andre Swift,RB,CHI,53.2,-114,0.71,NaN,NaN
5,JAX @ CIN,Sun 10:00 AM,Chase Brown,RB,CIN,44.0,+127,0.55,NaN,NaN
6,DAL @ HOU,Sun 10:00 AM,Javonte Williams,RB,DAL,63.8,-177,0.92,NaN,NaN
7,ARI @ NYG,Sun 10:00 AM,Cam Skattebo,RB,NYG,52.1,-109,0.70,NaN,NaN
8,LA @ PHI,Sun 10:00 AM,Kyren Williams,RB,LA,45.5,+120,0.58,NaN,NaN
9,GB @ TB,Sun 10:00 AM,Christian Watson,WR,GB,43.5,+130,0.55,NaN,NaN


## 6. Sportsbook props vs the model
Sorted by expected value per $1. The `book` column says where each line came from. Sources (`LINES` in section 1), all but the last free with no key:

| Source | Prices | Anytime TD | Attempts / completions / carries |
|---|---|---|---|
| **FanDuel** (`"fanduel"`) | Real | Yes | No |
| **ESPN** (`"espn"`, DraftKings' lines) | -110 assumed | No | Yes |
| **The Odds API** (`"odds_api"`, key needed) | Real DraftKings | Yes | Yes |

`"auto"` uses The Odds API if `odds_api_key.txt` exists, then FanDuel, then ESPN. Books post Sunday props by about Wednesday.

Huge edges usually mean the model is missing something (a new starter, a role change), not that the book is wrong.

In [21]:
e = report['edges'].copy()
if len(e):
    def _mean(r):
        if r.market == 'anytime_td':
            return float('nan')
        a, h = r.game.split(' @ ')
        d = report['sims'][(h, a)][1].draws.get((r.player, r.market))
        return float(d.mean()) if d is not None else float('nan')
    e.insert(e.columns.get_loc('model_median'), 'model_mean', e.apply(_mean, axis=1).round(1))
    e = e.drop(columns='model_median')
print(report.get('lines_note', ''))
e.head(40) if len(e) else print('No sportsbook lines found yet for this week.')

FanDuel lines and prices as of Mon Oct 05, 09:49 AM Arizona time (no attempts, completions or carries)


,player,game,book,market,line,model_mean,pick,model_prob,dk_price,fair_price,ev_per_unit
0,Kevin Austin Jr.,ATL @ NO,FanDuel,anytime_td,NaN,NaN,yes,0.1441,+1100,+594,0.729200
1,Drake London,ATL @ NO,FanDuel,receptions,5.5,4.0,under,0.7710,+106,-337,0.588260
2,Devaughn Vele,ATL @ NO,FanDuel,receptions,3.5,3.2,under,0.6095,+128,-156,0.389660
3,Drake London,ATL @ NO,FanDuel,rec_yds,79.5,59.6,under,0.7319,-113,-273,0.379631
4,Juwan Johnson,ATL @ NO,FanDuel,rec_yds,45.5,33.8,under,0.7280,-113,-268,0.372280
5,Chris Olave,ATL @ NO,FanDuel,receptions,6.5,5.5,under,0.6818,+100,-214,0.363600
6,Bijan Robinson,ATL @ NO,FanDuel,rec_yds,37.5,28.3,under,0.7231,-113,-261,0.363044
7,Alvin Kamara,ATL @ NO,FanDuel,rec_yds,14.5,11.1,under,0.7098,-113,-245,0.337973
8,Alvin Kamara,ATL @ NO,FanDuel,receptions,3.5,1.8,under,0.8720,-188,-681,0.335817
9,Juwan Johnson,ATL @ NO,FanDuel,receptions,3.5,3.3,under,0.5943,+118,-146,0.295574


## 7. Track record
Grades every logged prediction whose game has finished. Watch **closing line value** first: it becomes informative after a few dozen games, while win-loss records need hundreds.

In [16]:
track = grade_log()

Games: {'games graded': 15, 'winner %': '73.3%', 'spread miss (model)': np.float64(4.8), 'spread miss (market at log)': np.float64(6.2), 'ATS vs logged line': '8-4 (66.7%)', 'spread CLV (avg pts)': np.float64(0.07), 'beat the closing spread': '6.7% (moved away: 0.0%)', 'O/U vs logged total': '9-6 (60.0%)', 'total CLV (avg pts)': np.float64(0.07)}
Only 15 games graded: records this small are mostly luck. CLV becomes informative first.


## 8. Reference tables

In [17]:
ns.coach_table(report['model'])

,coach,4th_GOE_punt_zone,4th_GOE_fg_zone,halftime_off_swing,halftime_def_swing
team,,,,,
DAL,Brian Schottenheimer,0.128,0.042,-0.031,0.023
IND,Shane Steichen,0.093,0.014,-0.015,-0.026
CAR,Dave Canales,0.070,-0.007,0.028,0.027
BUF,Sean McDermott,0.064,0.153,0.020,0.049
DET,Dan Campbell,0.059,0.168,0.011,-0.038
ATL,Raheem Morris,0.042,0.007,-0.010,-0.022
KC,Andy Reid,0.042,-0.075,0.005,0.004
NYG,John Harbaugh,0.041,0.005,0.052,-0.039
PHI,Nick Sirianni,0.039,0.067,-0.064,-0.021


In [18]:
ns.qb_table(report['model']).head(32)

,name,team,db,epa_per_db,turnover_pct
qb_id,,,,,
00-0033873,P.Mahomes,KC,5447,0.222,1.878
00-0037834,B.Purdy,SF,1861,0.219,2.502
00-0034857,J.Allen,BUF,5000,0.183,2.275
00-0034796,L.Jackson,BAL,3580,0.169,2.213
00-0033077,D.Prescott,DAL,3904,0.146,2.238
00-0036442,J.Burrow,CIN,3535,0.120,2.143
00-0026498,M.Stafford,LA,4123,0.118,2.330
00-0039910,J.Daniels,WAS,1039,0.115,1.834
00-0033106,J.Goff,DET,4599,0.114,2.228


In [19]:
ns.active_referees(report['model'])

,games,flags_pg,away_minus_home_flags
referee,,,
Adrian Hill,117,12.69,0.62
Alan Eck,53,11.24,0.23
Alex Kemp,120,12.56,0.59
Alex Moore,19,13.83,0.62
Bill Vinovich,121,11.10,0.28
Brad Allen,117,11.61,0.62
Brad Rogers,111,12.36,0.09
Carl Cheffers,120,12.46,0.48
Clay Martin,116,11.83,0.41


## 9. Game backtest vs Vegas (optional, ~3 minutes per season)

In [22]:
RUN_BACKTEST = True
if RUN_BACKTEST:
    D = report["data"]
    bt, _ = ns.backtest(D["games"], D["prep"], D["injuries"], ns.load_rosters(2025)[0], 2025, n_sims=4000)
    display(ns.grade_vs_vegas(bt))

games                      224
winner_pct_model      0.627803
winner_pct_vegas      0.632287
spread_miss_model    10.224632
spread_miss_vegas     9.924107
total_miss_model     10.228775
total_miss_vegas        9.9375
ATS_record             111-112
ATS_pct               0.497758
ATS_p_value           0.553251
OU_record              118-106
OU_pct                0.526786
OU_p_value            0.231218
model_total_bias      1.107629
dtype: object